# 01 · Feature Engineering — LRFM-C 用户画像构建

## 1. 模块目标（Objective）

本模块的目标是基于清洗后的电商行为日志，构建**用户级特征表（LRFM-C）**，用于后续的用户分群、画像分析以及转化预测建模。

具体来说，本模块将完成：

- 构建稳定的商品维度表（item → category / availability）
- 将商品类目信息回填至用户行为日志
- 基于用户行为轨迹提取 **LRFM-C** 特征
- 过滤一次性/噪声用户，保留具有真实行为模式的活跃用户

---

## 2. 数据输入（Input Tables）

本模块使用的数据来自 `00_setup_and_data_load` 阶段生成的 Parquet 文件：

- **events_clean.parquet**  
  用户行为日志（view / addtocart / transaction）

- **item_properties_clean.parquet**  
  商品属性变更日志（包含 categoryid、available 等）

- **category_tree_clean.parquet**  
  商品类目层级结构（父子类目关系）

---

## 3. 关键设计决策（Key Design Choices）

### 3.1 商品维表构建（item_dim）

- 原始商品属性表为 **时间序列日志**（20M+ 行），同一商品属性可能随时间变化
- 本项目中：
  - 对每个 `(itemid, property)` 仅保留**最新记录**
  - 仅使用业务语义明确且稳定的属性：
    - `categoryid`
    - `available`

最终得到 **item_dim（~417k 行）**，作为稳定商品维度表。

---

### 3.2 类目缺失的处理策略

由于部分商品在行为表中出现，但未在属性日志中记录，导致：

- 约 **9.27% 行为记录**
- 约 **21.19% 商品**

无法映射到类目。

为兼顾**数据完整性**与**类目分析的干净口径**，本项目采用双口径策略：

- **events_enriched_all**  
  - 保留全部事件
  - 未知类目用 `categoryid = -1` 标记

- **events_enriched_cat**  
  - 仅保留可映射类目的事件
  - 用于类目分析与 LRFM-C 中的 C（Category）特征构建

---

### 3.3 活跃用户过滤（User Filtering）

EDA 显示用户行为存在明显长尾：

- 超过 50% 的用户仅发生 1 次行为
- 90% 用户行为数 ≤ 3

为避免噪声用户影响画像与建模效果，本模块后续分析仅保留：

> **至少发生 3 次行为的用户（n_events ≥ 3）**

---

## 4. LRFM-C 特征定义

基于过滤后的用户行为数据，构建以下特征：

### R — Recency
- 距观察期结束的最近一次行为天数

### F — Frequency
- 浏览次数（view）
- 加购次数（addtocart）
- 成交次数（transaction）

### M — Monetary（Proxy）
- 由于缺乏稳定的交易金额字段，使用 **成交次数（transaction_cnt）** 作为用户价值 proxy

### C — Category Preference
- 不同类目数量（`n_categories`）
- Top 类目行为占比（`top_category_share`），衡量用户偏好集中度

---

## 5. 模块输出（Outputs）

本模块将产出以下核心数据集：

- **item_dim.parquet**  
  商品维度表（item → category / availability）

- **events_enriched_cat.parquet**  
  带类目的用户行为日志（干净类目口径）

- **user_features_lrfmc.parquet**  
  用户级 LRFM-C 特征表（用于分群与建模）

- **user_features_for_clustering.parquet**  
  经过 log 变换与标准化的聚类输入特征

---

## 6. 后续分析方向（Next Steps）

基于本模块输出的用户特征表，后续将开展：

1. 用户分群（KMeans / HDBSCAN）
2. 用户画像与类目偏好分析
3. 运营策略与转化提升建议
4. （可选）转化预测 / 留存建模



In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path(r"E:\data_analysis\ecommece")

CLEAN_DIR_00 = BASE_DIR / "data_clean" / "00_setup_and_data_load"
CLEAN_DIR_01 = BASE_DIR / "data_clean" / "01_feature_engineering"
CLEAN_DIR_01.mkdir(parents=True, exist_ok=True)

events_path = CLEAN_DIR_00 / "events_clean.parquet"
props_path  = CLEAN_DIR_00 / "item_properties_clean.parquet"
cat_path    = CLEAN_DIR_00 / "category_tree_clean.parquet"

print("events_path:", events_path.exists(), events_path)
print("props_path :", props_path.exists(),  props_path)
print("cat_path   :", cat_path.exists(),    cat_path)

events_path: True E:\data_analysis\ecommece\data_clean\00_setup_and_data_load\events_clean.parquet
props_path : True E:\data_analysis\ecommece\data_clean\00_setup_and_data_load\item_properties_clean.parquet
cat_path   : True E:\data_analysis\ecommece\data_clean\00_setup_and_data_load\category_tree_clean.parquet


In [2]:
props = pd.read_parquet(
    props_path,
    columns=["itemid", "property", "value", "timestamp"] 
)

print("props shape:", props.shape)
print("props properties (top5):")
print(props["property"].value_counts().head(5))

need_props = ["categoryid", "available"]
p = props[props["property"].isin(need_props)].copy()

print("\nFiltered props shape:", p.shape)
print("Filtered property counts:\n", p["property"].value_counts())

p = p.sort_values(["itemid", "property", "timestamp"])
p_last = p.drop_duplicates(subset=["itemid", "property"], keep="last")

print("\nAfter taking latest per (itemid, property):", p_last.shape)

props shape: (20275902, 4)
props properties (top5):
property
888           3000398
790           1790516
available     1503639
categoryid     788214
6              631471
Name: count, dtype: Int64

Filtered props shape: (2291853, 4)
Filtered property counts:
 property
available     1503639
categoryid     788214
Name: count, dtype: Int64

After taking latest per (itemid, property): (834106, 4)


In [3]:
item_dim = (
    p_last
    .pivot(index="itemid", columns="property", values="value")
    .reset_index()
)

item_dim["categoryid"] = pd.to_numeric(item_dim["categoryid"], errors="coerce").astype("Int64")
item_dim["available"]  = pd.to_numeric(item_dim["available"],  errors="coerce").astype("Int64")

print("\nitem_dim shape:", item_dim.shape)
print("missing categoryid:", item_dim["categoryid"].isna().sum())
print("missing available :", item_dim["available"].isna().sum())

display(item_dim.head())

out_path = CLEAN_DIR_01 / "item_dim.parquet"
item_dim.to_parquet(out_path, index=False)
print("\nSaved:", out_path)


item_dim shape: (417053, 3)
missing categoryid: 0
missing available : 0


property,itemid,available,categoryid
0,0,0,209
1,1,0,1114
2,2,0,1305
3,3,0,1171
4,4,0,1038



Saved: E:\data_analysis\ecommece\data_clean\01_feature_engineering\item_dim.parquet


In [4]:
BASE_DIR = Path(r"E:\data_analysis\ecommece")

CLEAN_DIR_00 = BASE_DIR / "data_clean" / "00_setup_and_data_load"
CLEAN_DIR_01 = BASE_DIR / "data_clean" / "01_feature_engineering"

events_path = CLEAN_DIR_00 / "events_clean.parquet"
item_dim_path = CLEAN_DIR_01 / "item_dim.parquet"

events = pd.read_parquet(
    events_path,
    columns=["visitorid", "itemid", "event", "timestamp", "dt", "transactionid"]
)

item_dim = pd.read_parquet(item_dim_path)

print("events:", events.shape)
print("item_dim:", item_dim.shape)

events_enriched = events.merge(
    item_dim,
    on="itemid",
    how="left",
    validate="many_to_one"  # sanity check: many events -> one item
)

print("\nevents_enriched shape:", events_enriched.shape)
print("missing categoryid:", events_enriched["categoryid"].isna().sum())
print("missing available :", events_enriched["available"].isna().sum())

display(events_enriched.head())

events: (2755641, 6)
item_dim: (417053, 3)

events_enriched shape: (2755641, 8)
missing categoryid: 255576
missing available : 255576


,visitorid,itemid,event,timestamp,dt,transactionid,available,categoryid
0,257597,355908,view,1433221332117,2015-06-02 05:02:12.117000+00:00,<NA>,1,1173
1,992329,248676,view,1433224214164,2015-06-02 05:50:14.164000+00:00,<NA>,1,1231
2,111016,318965,view,1433221999827,2015-06-02 05:13:19.827000+00:00,<NA>,<NA>,<NA>
3,483717,253185,view,1433221955914,2015-06-02 05:12:35.914000+00:00,<NA>,0,914
4,951259,367447,view,1433221337106,2015-06-02 05:02:17.106000+00:00,<NA>,0,491


In [5]:
n_events = len(events_enriched)
n_missing_rows = events_enriched["categoryid"].isna().sum()

n_event_items = events_enriched["itemid"].nunique()
n_dim_items = item_dim["itemid"].nunique()

missing_item_ids = events_enriched.loc[events_enriched["categoryid"].isna(), "itemid"].unique()
n_missing_items = len(missing_item_ids)

print("Missing rows:", n_missing_rows, f"({n_missing_rows/n_events:.2%} of events)")
print("Unique itemids in events:", n_event_items)
print("Unique itemids in item_dim:", n_dim_items)
print("Missing unique itemids:", n_missing_items, f"({n_missing_items/n_event_items:.2%} of event items)")

Missing rows: 255576 (9.27% of events)
Unique itemids in events: 235061
Unique itemids in item_dim: 417053
Missing unique itemids: 49815 (21.19% of event items)


In [6]:
events_all = events_enriched.copy()
events_all["categoryid"] = events_all["categoryid"].fillna(-1).astype("int64")
events_all["available"]  = events_all["available"].fillna(-1).astype("int64")

events_cat = events_enriched.dropna(subset=["categoryid", "available"]).copy()
events_cat["categoryid"] = events_cat["categoryid"].astype("int64")
events_cat["available"]  = events_cat["available"].astype("int64")

print("events_all:", events_all.shape, "| unknown rows:", (events_all["categoryid"] == -1).sum())
print("events_cat:", events_cat.shape)

out_all = CLEAN_DIR_01 / "events_enriched_all.parquet"
out_cat = CLEAN_DIR_01 / "events_enriched_cat.parquet"

events_all.to_parquet(out_all, index=False)
events_cat.to_parquet(out_cat, index=False)

print("Saved:")
print(" -", out_all)
print(" -", out_cat)

events_all: (2755641, 8) | unknown rows: 255576
events_cat: (2500065, 8)
Saved:
 - E:\data_analysis\ecommece\data_clean\01_feature_engineering\events_enriched_all.parquet
 - E:\data_analysis\ecommece\data_clean\01_feature_engineering\events_enriched_cat.parquet


In [7]:
events_cat = pd.read_parquet(CLEAN_DIR_01 / "events_enriched_cat.parquet")

events_cat["week"] = events_cat["dt"].dt.to_period("W").astype(str)

weekly = (
        events_cat
        .groupby(["week", "event"])
        .size()
        .rename("count")
        .reset_index()
        .sort_values(["week", "event"])
)

print("=== Weekly event counts (head) ===")
display(weekly.head(5))

cat_funnel = (
    events_cat
    .groupby(["categoryid", "event"])
    .size()
    .unstack(fill_value = 0)
)

if "view" in cat_funnel.columns and "transaction" in cat_funnel.columns:
    cat_funnel["txn_per_view"] = cat_funnel["transaction"] / cat_funnel["view"].replace(0, pd.NA)

if "addtocart" in cat_funnel.columns and "transaction" in cat_funnel.columns:
    cat_funnel["txn_per_cart"] = cat_funnel["transaction"] / cat_funnel["addtocart"].replace(0, pd.NA)

cat_funnel = cat_funnel.reset_index()

print("=== Top categories by views ===")
display(cat_funnel.sort_values("view", ascending=False).head(10))

print("=== Top categories by txn_per_view (min view>=200) ===")
display(cat_funnel[cat_funnel["view"] >= 200].sort_values("txn_per_view", ascending=False).head(10))

user_cnt = events_cat.groupby("visitorid").size().rename("n_events").reset_index()


print("=== User activity distribution (quantiles) ===")
display(user_cnt["n_events"].describe(percentiles=[.5,.75,.9,.95,.99]))

weekly.to_csv(CLEAN_DIR_01 / "eda_weekly_counts.csv", index=False, encoding="utf-8-sig")
cat_funnel.to_csv(CLEAN_DIR_01 / "eda_category_funnel.csv", index=False, encoding="utf-8-sig")
user_cnt.to_csv(CLEAN_DIR_01 / "eda_user_event_counts.csv", index=False, encoding="utf-8-sig")

print("Saved EDA outputs to:", CLEAN_DIR_01)

C:\Users\llj68\AppData\Local\Temp\ipykernel_20448\3851552659.py:3: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  events_cat["week"] = events_cat["dt"].dt.to_period("W").astype(str)


=== Weekly event counts (head) ===


,week,event,count
0,2015-04-27/2015-05-03,addtocart,288
1,2015-04-27/2015-05-03,transaction,80
2,2015-04-27/2015-05-03,view,12082
3,2015-05-04/2015-05-10,addtocart,3405
4,2015-05-04/2015-05-10,transaction,1077


=== Top categories by views ===


event,categoryid,addtocart,transaction,view,txn_per_view,txn_per_cart
676,1051,1709,485,73014,0.006643,0.283792
961,1483,1520,469,62785,0.007470,0.308553
322,491,1548,263,59298,0.004435,0.169897
615,959,1633,532,50112,0.010616,0.325781
222,342,1278,281,45288,0.006205,0.219875
445,683,907,211,37765,0.005587,0.232635
823,1279,720,123,33531,0.003668,0.170833
4,5,576,186,28952,0.006424,0.322917
426,646,484,87,27669,0.003144,0.179752
32,48,600,146,26701,0.005468,0.243333


=== Top categories by txn_per_view (min view>=200) ===


event,categoryid,addtocart,transaction,view,txn_per_view,txn_per_cart
1080,1685,65,34,419,0.081146,0.523077
93,147,39,21,290,0.072414,0.538462
37,57,128,150,2229,0.067295,1.171875
993,1541,55,13,216,0.060185,0.236364
221,340,18,11,204,0.053922,0.611111
288,444,65,35,651,0.053763,0.538462
68,103,19,12,236,0.050847,0.631579
689,1074,60,19,374,0.050802,0.316667
519,797,30,13,294,0.044218,0.433333
666,1037,652,226,5189,0.043554,0.346626


=== User activity distribution (quantiles) ===


count    1.236032e+06
mean     2.022654e+00
std      1.227991e+01
min      1.000000e+00
50%      1.000000e+00
75%      2.000000e+00
90%      3.000000e+00
95%      5.000000e+00
99%      1.300000e+01
max      6.643000e+03
Name: n_events, dtype: float64

Saved EDA outputs to: E:\data_analysis\ecommece\data_clean\01_feature_engineering


In [8]:
events_cat = pd.read_parquet(CLEAN_DIR_01 / "events_enriched_cat.parquet")

user_cnt = events_cat.groupby("visitorid").size().rename("n_events")
active_users = user_cnt[user_cnt >= 3].index

events_f = events_cat[events_cat["visitorid"].isin(active_users)].copy()
print("Filtered events:", events_f.shape)
print("Active users:", events_f["visitorid"].nunique())

events_f["categoryid"] = events_f["categoryid"].astype("int64")

obs_end = events_f["dt"].max()

last_event = events_f.groupby("visitorid")["dt"].max()
R = (obs_end - last_event).dt.days.rename("recency_days")

F = (
    events_f
    .groupby(["visitorid", "event"])
    .size()
    .unstack(fill_value = 0)
)

for col in ["view", "addtocart", "transaction"]:
    if col not in F.columns:
        F[col] = 0

F["transaction_cnt"] = F["transaction"]

cat_stats = (
    events_f
    .groupby(["visitorid", "categoryid"])
    .size()
    .rename("cnt")
    .reset_index()
)

cat_div = cat_stats.groupby("visitorid")["categoryid"].nunique().rename("n_categories")

top_cat_share = (
    cat_stats
    .sort_values(["visitorid", "cnt"], ascending = [True, False])
    .drop_duplicates("visitorid")
    .set_index("visitorid")["cnt"] /
    cat_stats.groupby("visitorid")["cnt"].sum()
)

top_cat_share = top_cat_share.rename("top_category_share")

user_features = pd.concat(
    [R, F[["view", "addtocart", "transaction", "transaction_cnt"]],
    cat_div, top_cat_share],
    axis = 1
).reset_index()

print("user_features shape:", user_features.shape)
display(user_features.head(20))

Filtered events: (1266663, 8)
Active users: 187481
user_features shape: (187481, 8)


,visitorid,recency_days,view,addtocart,transaction,transaction_cnt,n_categories,top_category_share
0,0,6,3,0,0,0,3,0.333333
1,2,41,8,0,0,0,2,0.750000
2,6,17,5,1,0,0,1,1.000000
3,7,124,3,0,0,0,2,0.666667
4,23,98,3,0,0,0,2,0.666667
5,37,35,8,0,0,0,2,0.875000
6,51,12,6,0,0,0,2,0.833333
7,54,0,11,0,0,0,2,0.909091
8,60,89,3,0,0,0,1,1.000000
9,64,86,7,0,0,0,3,0.571429


In [9]:
EVENTS_PATH = r"E:\data_analysis\ecommece\data_clean\01_feature_engineering\events_enriched_all.parquet"

events = pd.read_parquet(EVENTS_PATH)
events.head()

,visitorid,itemid,event,timestamp,dt,transactionid,available,categoryid
0,257597,355908,view,1433221332117,2015-06-02 05:02:12.117000+00:00,<NA>,1,1173
1,992329,248676,view,1433224214164,2015-06-02 05:50:14.164000+00:00,<NA>,1,1231
2,111016,318965,view,1433221999827,2015-06-02 05:13:19.827000+00:00,<NA>,-1,-1
3,483717,253185,view,1433221955914,2015-06-02 05:12:35.914000+00:00,<NA>,0,914
4,951259,367447,view,1433221337106,2015-06-02 05:02:17.106000+00:00,<NA>,0,491


In [13]:
USER_COL = "visitorid"
EVENT_COL = "event"
DT_COL = "dt"

In [15]:
user_agg = events.groupby(USER_COL).agg(
    events_total=(DT_COL, "size"),
    active_days=("date", "nunique"),
    first_dt=(DT_COL, "min"),
    last_dt=(DT_COL, "max"),
)

user_agg["lifetime_days"] = (
    user_agg["last_dt"] - user_agg["first_dt"]
).dt.days.clip(lower=0)

In [16]:
event_cnt = (
    events
    .pivot_table(
        index=USER_COL,
        columns=EVENT_COL,
        values=DT_COL,
        aggfunc="count",
        fill_value=0
    )
)

for c in ["view", "addtocart", "transaction"]:
    if c not in event_cnt.columns:
        event_cnt[c] = 0

user_agg = user_agg.join(event_cnt[["view", "addtocart", "transaction"]])

In [17]:
user_agg["behavior_density"] = (
    user_agg["events_total"] /
    user_agg["active_days"].replace(0, np.nan)
)

user_agg["addtocart_rate"] = (
    user_agg["addtocart"] /
    user_agg["view"].replace(0, np.nan)
)

user_agg["purchase_rate"] = (
    user_agg["transaction"] /
    user_agg["addtocart"].replace(0, np.nan)
)


In [18]:
end_dt = events[DT_COL].max()
start_30d = end_dt - pd.Timedelta(days=30)

events_30d = events[events[DT_COL] >= start_30d].copy()
events_30d["date"] = events_30d[DT_COL].dt.date

feat_30d = events_30d.groupby(USER_COL).agg(
    active_days_30d=("date", "nunique"),
)

txn_30d = (
    events_30d
    .loc[events_30d[EVENT_COL] == "transaction"]
    .groupby(USER_COL)
    .size()
    .rename("txn_30d")
)

feat_30d = feat_30d.join(txn_30d, how="left").fillna({"txn_30d": 0})

In [19]:
new_features = (
    user_agg
    .join(feat_30d, how="left")
    .fillna({"active_days_30d": 0, "txn_30d": 0})
    .reset_index()
)

new_features = new_features[[
    "visitorid",
    "active_days",
    "lifetime_days",
    "events_total",
    "behavior_density",
    "addtocart_rate",
    "purchase_rate",
    "active_days_30d",
    "txn_30d"
]]

new_features.head()

,visitorid,active_days,lifetime_days,events_total,behavior_density,addtocart_rate,purchase_rate,active_days_30d,txn_30d
0,0,1,0,3,3.0,0.0,NaN,1.0,0.0
1,1,1,0,1,1.0,0.0,NaN,0.0,0.0
2,2,1,0,8,8.0,0.0,NaN,0.0,0.0
3,3,1,0,1,1.0,0.0,NaN,0.0,0.0
4,4,1,0,1,1.0,0.0,NaN,1.0,0.0


In [22]:
USER_FEAT_PATH = r"E:\data_analysis\ecommece\data_clean\01_feature_engineering\user_features_lrfmc.parquet"
user_feat = pd.read_parquet(USER_FEAT_PATH)

import shutil
shutil.copy(USER_FEAT_PATH, USER_FEAT_PATH.replace(".parquet", "_backup.parquet"))

'E:\\data_analysis\\ecommece\\data_clean\\01_feature_engineering\\user_features_lrfmc_backup.parquet'

In [23]:
user_feat_updated = user_feat.merge(new_features, on="visitorid", how="left")


zero_cols = ["active_days", "lifetime_days", "events_total", "active_days_30d", "txn_30d"]
for c in zero_cols:
    if c in user_feat_updated.columns:
        user_feat_updated[c] = user_feat_updated[c].fillna(0)

rate_cols = ["behavior_density", "addtocart_rate", "purchase_rate"]
for c in rate_cols:
    if c in user_feat_updated.columns:
        user_feat_updated[c] = user_feat_updated[c].fillna(0)

user_feat_updated.to_parquet(USER_FEAT_PATH, index=False)
print("Overwritten:", USER_FEAT_PATH)

user_feat_updated.head()

Overwritten: E:\data_analysis\ecommece\data_clean\01_feature_engineering\user_features_lrfmc.parquet


,visitorid,recency_days,view,addtocart,transaction,transaction_cnt,n_categories,top_category_share,active_days,lifetime_days,events_total,behavior_density,addtocart_rate,purchase_rate,active_days_30d,txn_30d
0,0,6,3,0,0,0,3,0.333333,1,0,3,3.0,0.0,0.0,1.0,0.0
1,2,41,8,0,0,0,2,0.750000,1,0,8,8.0,0.0,0.0,0.0,0.0
2,6,17,5,1,0,0,1,1.000000,2,0,6,3.0,0.2,0.0,2.0,0.0
3,7,124,3,0,0,0,2,0.666667,2,1,3,1.5,0.0,0.0,0.0,0.0
4,23,98,3,0,0,0,2,0.666667,2,4,3,1.5,0.0,0.0,0.0,0.0
